# Whisper Baseline WER Test - Para Harcamadan Once Dogrulama

Bu notebook whisper-large-v3'u **fine-tune etmeden** Kuran ses verisi uzerinde test eder.

**Amac**: Egitim icin $37 harcamadan once modelin potansiyelini gormek.

**Mantik**:
- Sifir fine-tune ile WER %8-12 → LoRA ile %3'e dusme ihtimali cok yuksek ✅
- Sifir fine-tune ile WER %15+ → Bir sorun var, once arastir ⚠️

**Gereksinimler**: Colab ucretsiz T4 GPU (sadece inference, egitim yok)

**Sure**: ~5-10 dakika

## 1. Kurulum

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip install -q transformers datasets jiwer soundfile librosa tqdm accelerate

## 2. Imports & Normalizasyon

In [ ]:
import re
import torch
import jiwer
from tqdm import tqdm
from datasets import load_dataset
from transformers import WhisperForConditionalGeneration, WhisperProcessor

# Arapca normalizasyon
DIACRITICS = re.compile("[\u064b-\u0652\u0653-\u0655\u0670]")

def normalize_arabic(text: str) -> str:
    text = DIACRITICS.sub("", text)
    text = text.replace("\u0640", "")   # tatweel
    text = text.replace("\u0622", "\u0627")  # alef madda
    text = text.replace("\u0623", "\u0627")  # alef hamza above
    text = text.replace("\u0625", "\u0627")  # alef hamza below
    text = text.replace("\u0671", "\u0627")  # alef wasla
    text = text.replace("\u0624", "\u0648")  # waw hamza
    text = text.replace("\u0626", "\u064a")  # yeh hamza
    text = text.replace("\u0629", "\u0647")  # ta marbuta
    text = text.replace("\u0649", "\u064a")  # alef maksura
    text = re.sub(r"\s+", " ", text).strip()
    return text

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Test: ", normalize_arabic("بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ"))

## 3. Test Edilecek Modeller

3 modeli karsilastiriyoruz:
- `whisper-small` (244M) - bizim ilk plan
- `whisper-large-v3` (1.5B) - yeni plan
- `tarteel-ai/whisper-base-ar-quran` (77M, fine-tuned) - rakip benchmark

In [ ]:
MODELS = {
    "whisper-small (244M, no fine-tune)": "openai/whisper-small",
    "whisper-large-v3 (1.5B, no fine-tune)": "openai/whisper-large-v3",
    "tarteel-ai/whisper-base-ar-quran (77M, fine-tuned)": "tarteel-ai/whisper-base-ar-quran",
}

# Kac ornek test edelim (daha fazla = daha guvenilir ama daha yavas)
N_SAMPLES = 100

print(f"{len(MODELS)} model, {N_SAMPLES} ornek ile test edilecek")

## 4. Dataset Yukle (Sadece Test Split)

In [ ]:
print("Dataset yukleniyor (streaming)...")
dataset = load_dataset("tarteel-ai/everyayah", split="test", streaming=True)

# N_SAMPLES ornek al
test_samples = []
for i, sample in enumerate(dataset):
    if i >= N_SAMPLES:
        break
    # 30s filtreleme
    duration = sample["audio"]["array"].shape[0] / sample["audio"]["sampling_rate"]
    if duration <= 30:
        test_samples.append(sample)

print(f"{len(test_samples)} test ornegi yuklendi")

# Ornek goster
print(f"\nIlk ornek:")
print(f"  Metin: {test_samples[0]['text'][:80]}...")
dur = test_samples[0]['audio']['array'].shape[0] / test_samples[0]['audio']['sampling_rate']
print(f"  Sure: {dur:.1f}s")

## 5. Baseline WER Testi

In [ ]:
results = {}

for model_name, model_id in MODELS.items():
    print(f"\n{'='*60}")
    print(f"Test: {model_name}")
    print(f"{'='*60}")

    # Model yukle
    print(f"Model yukleniyor: {model_id}")
    processor = WhisperProcessor.from_pretrained(model_id)
    model = WhisperForConditionalGeneration.from_pretrained(
        model_id, torch_dtype=torch.float16, device_map="auto"
    )
    model.eval()

    # Generation config temizle - eski forced_decoder_ids kalintisi olmasin
    model.generation_config.forced_decoder_ids = None
    model.generation_config.suppress_tokens = []

    # OpenAI modelleri icin Arapca dil ayari
    # Tarteel gibi fine-tuned modellerde lang_to_id olmayabilir, try/except ile koru
    if "openai" in model_id:
        try:
            model.generation_config.language = "ar"
            model.generation_config.task = "transcribe"
        except Exception as e:
            print(f"  Language config atanamadi: {e}")

    predictions = []
    references = []

    for sample in tqdm(test_samples, desc="Inference"):
        audio = sample["audio"]
        ref_text = sample["text"]

        inputs = processor(
            audio["array"],
            sampling_rate=audio["sampling_rate"],
            return_tensors="pt",
        )

        with torch.no_grad():
            try:
                predicted_ids = model.generate(
                    input_features=inputs.input_features.to(device=model.device, dtype=torch.float16),
                    max_new_tokens=225,
                )
            except Exception as e:
                # fp16 basarisiz olursa fp32 dene
                print(f"  fp16 hatasi, fp32 deneniyor: {e}")
                predicted_ids = model.generate(
                    input_features=inputs.input_features.to(device=model.device),
                    max_new_tokens=225,
                )

        pred_text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        predictions.append(pred_text)
        references.append(ref_text)

    # WER hesapla
    wer_diac = 100 * jiwer.wer(references, predictions)

    refs_norm = [normalize_arabic(r) for r in references]
    preds_norm = [normalize_arabic(p) for p in predictions]
    wer_norm = 100 * jiwer.wer(refs_norm, preds_norm)

    results[model_name] = {
        "wer_diacritized": round(wer_diac, 2),
        "wer_normalized": round(wer_norm, 2),
    }

    print(f"\nSonuclar:")
    print(f"  WER (diacritized): {wer_diac:.2f}%")
    print(f"  WER (normalized):  {wer_norm:.2f}%")

    print(f"\nOrnek ciktilar:")
    for i in range(min(3, len(predictions))):
        print(f"  Ref:  {references[i][:60]}")
        print(f"  Pred: {predictions[i][:60]}")
        sample_wer = 100 * jiwer.wer(references[i], predictions[i])
        print(f"  WER:  {sample_wer:.1f}%")
        print()

    del model, processor
    torch.cuda.empty_cache()

print("\nTum testler tamamlandi!")

## 6. Sonuc Tablosu & Karar

In [ ]:
print("\n" + "="*70)
print("                    SONUC TABLOSU")
print("="*70)
print(f"{'Model':<50} {'WER(diac)':<12} {'WER(norm)'}")
print("-"*70)

for model_name, res in results.items():
    print(f"{model_name:<50} {res['wer_diacritized']:<12} {res['wer_normalized']}")

print("-"*70)

# Karar
large_v3_wer = results.get("whisper-large-v3 (1.5B, no fine-tune)", {}).get("wer_normalized", 999)
tarteel_wer = results.get("tarteel-ai/whisper-base-ar-quran (77M, fine-tuned)", {}).get("wer_normalized", 999)

print("\n" + "="*70)
print("                    KARAR")
print("="*70)

if large_v3_wer < 15:
    print(f"\n✅ whisper-large-v3 fine-tune OLMADAN WER: {large_v3_wer}%")
    print(f"   Bu cok iyi! LoRA ile %3'e dusme ihtimali cok yuksek.")
    print(f"   Runpod'da egitim icin $37 harcamaya DEGER.")
    if large_v3_wer < tarteel_wer:
        print(f"\n🏆 BONUS: Fine-tune olmadan bile Tarteel'den ({tarteel_wer}%) iyi!")
        print(f"   LoRA ile %2-3 WER hedefi cok gercekci.")
    else:
        fark = large_v3_wer - tarteel_wer
        print(f"\n📊 Tarteel ({tarteel_wer}%) su an {fark:.1f}% daha iyi.")
        print(f"   Ama onlar fine-tuned, biz degiliz. LoRA ile geceriz.")
elif large_v3_wer < 25:
    print(f"\n⚠️ whisper-large-v3 fine-tune OLMADAN WER: {large_v3_wer}%")
    print(f"   Orta seviye. LoRA ile %5 altina inme ihtimali var ama %3 zor.")
    print(f"   Oneri: Oncelikle whisper-small ile ucuz deneme yap.")
else:
    print(f"\n❌ whisper-large-v3 fine-tune OLMADAN WER: {large_v3_wer}%")
    print(f"   Bir sorun var. Veriyi veya ayarlari kontrol et.")
    print(f"   $37 harcama, once sorunu bul.")

print("\n" + "="*70)